# Data Analysis and PreProcessing

## Setting up Environment

In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYSPARK_PYTHON'] = '/home/subha/miniconda3/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/home/subha/miniconda3/bin/python'

## Starting Pyspark with Master 10G Memory

In [2]:
import findspark
findspark.init('/home/subha/aiwork/spark')
# Initializing the spark context
#import pyspark.pandas as ps
#pdf_incidents = df_incidents.to_pandas_on_spark()
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col, lower, regexp_replace
from pyspark.sql.types import StringType, ArrayType
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from pyspark.sql.functions import *

# Configure Spark to use multiple threads
spark = SparkSession.builder.appName("Amazon Reviews Analysis")\
    .master("local[*]")\
    .config("spark.executorEnv.PYSPARK_PYTHON", "/home/subha/miniconda3/bin/python")\
    .config("spark.driver.maxResultSize","10g")\
    .config("spark.executor.instances", "4")\
    .config("spark.executor.cores", "2")\
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

25/03/23 11:50:38 WARN Utils: Your hostname, neoshiva resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/23 11:50:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/23 11:50:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Gathering Train and Test Data

### Reading Test and Train Datasets

In [3]:
df_test = spark.read.text("amazon_train_dataset/test.ft.txt")
df_train = spark.read.text("amazon_train_dataset/train.ft.txt")

### Shaping the Datasets (Feature Split)

In [4]:
# Show the first few rows to understand the data format
def shaping_datasets(df):
    leng = len("__label__1")
    # Split the text into two parts: label and review text
    df_split = df.withColumn("label", split(col("value"), " ").getItem(0))\
    .withColumn("review_text", substring(df.value,12,10000))  # get review text from the second word onward)
    
    # Map label to sentiment: "negative" for __label__1, "positive" for __label__2
    # df_sentiment_reviews = df_split.withColumn(
    #     "sentiment",
    #     when(col("label") == "__label__1", "negative")
    #     .when(col("label") == "__label__2", "positive")
    #     .otherwise("unknown")
    # )
    df_sentiment_reviews = df_split.withColumn(
        "sentiment",
        when(col("label") == "__label__1", 0)
        .when(col("label") == "__label__2", 1)
        .otherwise("unknown")
    )
    
    # Select only the relevant columns: sentiment and review text
    df_sentiment_reviews = df_sentiment_reviews.select("sentiment", "review_text")
    
    return df_sentiment_reviews

## Calling the Function
df_test_p = shaping_datasets(df_test)
df_train_p = shaping_datasets(df_train)

### Display and Counts

In [5]:
display(df_train_p.show(5))

+---------+--------------------+
|sentiment|         review_text|
+---------+--------------------+
|        1|Stuning even for ...|
|        1|The best soundtra...|
|        1|Amazing!: This so...|
|        1|Excellent Soundtr...|
|        1|Remember, Pull Yo...|
+---------+--------------------+
only showing top 5 rows



None

In [6]:
df_train_p.count()

3600000

In [7]:
df_test_p.show(10)

+---------+--------------------+
|sentiment|         review_text|
+---------+--------------------+
|        1|Great CD: My love...|
|        1|One of the best g...|
|        0|Batteries died wi...|
|        1|works fine, but M...|
|        1|Great for the non...|
|        0|DVD Player crappe...|
|        0|Incorrect Disc: I...|
|        0|DVD menu select p...|
|        1|Unique Weird Orie...|
|        0|Not an "ultimate ...|
+---------+--------------------+
only showing top 10 rows



In [7]:
df_test_p.count()

400000

## Natural Language Preprocessing

### NLTK resources

In [8]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/subha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/subha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/subha/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/subha/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

### Lemmetizer and Stop Words

In [9]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

### Clean, Tokenize and UDF creation

In [10]:
def clean_text(text):
    """
    Clean text by removing special characters, numbers, and converting to lowercase
    """
    if not text:
        return text
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def tokenize_and_preprocess(text):
    """
    Tokenize, remove stopwords, and lemmatize text
    """
    if not text:
        return []
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize
    # tokens = [lemmatizer.lemmatize(token) for token in tokens 
    #          if token.lower() not in stop_words and len(token) > 2]
    tokens = [token for token in tokens 
             if token.lower() not in stop_words and len(token) > 2]
    
    return tokens
    
# Register UDFs
clean_text_udf = udf(clean_text, StringType())
tokenize_and_preprocess_udf = udf(tokenize_and_preprocess, ArrayType(StringType()))

### NLTK Review Process Method

In [11]:
def process_reviews_with_nltk(reviews_df):
    """
    Process reviews using NLTK for text cleaning and preprocessing
    
    Args:
        reviews_df: DataFrame with 'sentiment' and 'reviews' columns
    Returns:
        DataFrame with processed text
    """
    
    # Apply text cleaning
    processed_df = reviews_df.withColumn(
        "cleaned_text",
        clean_text_udf(col("review_text"))
    )
    
    # Apply tokenization, stopword removal, and lemmatization
    processed_df = processed_df.withColumn(
        "processed_tokens",
        tokenize_and_preprocess_udf(col("cleaned_text"))
    )
    
    # Convert tokens back to text
    processed_df = processed_df.withColumn(
        "processed_text",
        udf(lambda x: ' '.join(x) if x else '', StringType())(col("processed_tokens"))
    )
    
    return processed_df

## Executing the Preprocessing Steps

### Test and Train Spark DF created

In [12]:
test_processed_reviews_df = process_reviews_with_nltk(df_test_p)
train_processed_reviews_df = process_reviews_with_nltk(df_train_p)
test_processed_reviews_df.printSchema()

root
 |-- sentiment: string (nullable = false)
 |-- review_text: string (nullable = true)
 |-- cleaned_text: string (nullable = true)
 |-- processed_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- processed_text: string (nullable = true)



### Writing to Output Location

In [13]:
test_processed_reviews_df.select("cleaned_text","sentiment").coalesce(1).write.mode("overwrite").parquet("output/cleandata/large/test_data")

In [14]:
train_processed_reviews_df.select("cleaned_text","sentiment").coalesce(1).write.mode("overwrite").parquet("output/cleandata/large/train_data")

### Creating Samples for hyper Parameter Tuning

#### Train Data

In [15]:
train_processed_reviews_df.createOrReplaceTempView("train_processed_reviews_df")
test_processed_reviews_df.createOrReplaceTempView("test_processed_reviews_df")

In [16]:
sql = """
select * 
from train_processed_reviews_df 
where sentiment = 0  limit 180000
"""
tr_df_1 = spark.sql(sql)

sql ="""
select * 
from train_processed_reviews_df 
where sentiment = 1  limit 180000
"""
tr_df_2 = spark.sql(sql)

u_tr_df = tr_df_1.union(tr_df_2)

In [17]:
u_tr_df.select("cleaned_text","sentiment").coalesce(5).write.mode("overwrite").parquet("output/cleandata/large/train_data_sample")

#### Test and Validation

In [18]:
sql = """
select * 
from test_processed_reviews_df 
where sentiment = 0  limit 20000
"""
te_df_1 = spark.sql(sql)
val_df_1, test_df_1 = te_df_1.randomSplit([0.5, 0.5]) 

sql1 ="""
select * 
from test_processed_reviews_df 
where sentiment = 1  limit 20000
"""
te_df_2 = spark.sql(sql1)
val_df_2, test_df_2 = te_df_2.randomSplit([0.5, 0.5]) 

u_val_te_df = val_df_1.union(val_df_2)
u_test_te_df = test_df_1.union(test_df_2)

In [19]:
print(val_df_1.count(),val_df_2.count(),test_df_1.count(),test_df_2.count())

[Stage 21:====================================================>   (30 + 2) / 32]

10075 10158 9925 9842


In [20]:
u_val_te_df.select("cleaned_text","sentiment").coalesce(5).write.mode("overwrite").parquet("output/cleandata/large/val_data_sample")
u_test_te_df.select("cleaned_text","sentiment").coalesce(5).write.mode("overwrite").parquet("output/cleandata/large/test_data_sample")

## Appendix

In [33]:
spark.stop()